# Request analysis (task 4a) on Kaggle

RealEdit's simplification and taxonomy prompts in one call per request, with an open VLM (Qwen2.5-VL).

Before running:
- input: the private dataset `editjudge-data`;
- settings: GPU T4 x2 (7B model) or one T4 (3B model), internet on;
- the code is cloned from GitHub (`BRANCH`), so push your changes first;
- get the coordinator's approval for the run.

To resume a run in a new session, add the previous notebook output as an input and set `PREVIOUS`: its
`instructions/` folder is copied into `OUT` and done requests are skipped.

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"                            # or your task branch, e.g. "instructions/hai"
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.environ["HF_HOME"] = "/kaggle/working/hf"   # model weights cache (deleted at the end)
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

PREVIOUS = None                            # e.g. "/kaggle/input/editjudge-instructions-run1/out"
if PREVIOUS:
    shutil.copytree(f"{PREVIOUS}/instructions", f"{OUT}/instructions", dirs_exist_ok=True)

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate
!python -m src.instructions list

In [ ]:
# 1. Smoke test: 20 requests with the small model
!python -m src.instructions run --client qwen2.5-vl-3b --sample 20
!python -m src.instructions stats --run qwen2.5-vl-3b:analyze_v1

In [ ]:
# 2. Prompt tuning: the same fixed 100 requests for every model and prompt version
# !python -m src.instructions run --client qwen2.5-vl-7b --sample 100
# !python -m src.instructions agree --run qwen2.5-vl-3b:analyze_v1 --run qwen2.5-vl-7b:analyze_v1

In [ ]:
# 3. Full run (17,755 requests; split over sessions if needed)
# !python -m src.instructions run --client qwen2.5-vl-7b

In [ ]:
shutil.rmtree("/kaggle/working/hf", ignore_errors=True)   # keep only the results
!ls -R /kaggle/working/out | head